In [ ]:
%pip -q install requests pillow

import os, json, hashlib, zipfile, stat, shutil, math, time
from pathlib import Path, PurePosixPath
from urllib.parse import urlparse
import requests
from google.colab import userdata

# Source: the two original ZIP assets in the existing GitHub release.
SOURCE_REPO = "ZzwahwahzZ/Conquer-Mobile-Assets."
SOURCE_TAG = "mobile-source-files"

# Destination: publish the extracted files as downloadable ZIP bundles
# in a NEW GitHub Release in the same repository.
DEST_REPO = SOURCE_REPO
DEST_TAG = "mobile-extracted-assets"
DEST_RELEASE_NAME = "Conquer Mobile - Extracted Source Assets"

FILES = [
    {
        "name": "ConquerOnline.zip",
        "url": "https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets./releases/download/mobile-source-files/ConquerOnline.zip",
        "size": 461117324,
        "sha256": "b1c6958109a07c237631943fc0442d3161563d89e1020b143d95437ba9a6fdb0",
    },
    {
        "name": "Conquer-Mobile-Assets.-mobile-source-files.zip",
        "url": "https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets./releases/download/mobile-source-files/Conquer-Mobile-Assets.-mobile-source-files.zip",
        "size": 250389993,
        "sha256": "5acb29a128f4560bd84a956a628411f12ee713ceb5e2ecb935f65d31ba37d4e2",
    },
]

BASE = Path("/content/conquer_mobile_work")
DOWNLOADS = BASE / "downloads"
EXTRACTED = BASE / "extracted"
BUNDLES = BASE / "github_release_bundles"
REPORT_PATH = BASE / "extraction-report.json"
GITHUB_API = "https://api.github.com"
# Keep every release asset comfortably below GitHub's 2 GB per-asset ceiling.
MAX_BUNDLE_BYTES = 1500 * 1024 * 1024
SAFETY_BYTES = 512 * 1024 * 1024

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_verified(item, destination):
    h = hashlib.sha256()
    total = 0
    with requests.get(item["url"], stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with open(destination, "wb") as out:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    out.write(chunk)
                    h.update(chunk)
                    total += len(chunk)
    if total != item["size"]:
        raise RuntimeError(f'{item["name"]}: size mismatch; expected {item["size"]}, got {total}')
    digest = h.hexdigest()
    if digest.lower() != item["sha256"]:
        raise RuntimeError(f'{item["name"]}: SHA-256 mismatch: {digest}')
    return total, digest

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    count, total_bytes = 0, 0
    with zipfile.ZipFile(zip_path) as archive:
        bad = archive.testzip()
        if bad:
            raise RuntimeError(f"ZIP CRC test failed at: {bad}")
        for info in archive.infolist():
            raw = info.filename.replace("\\", "/")
            member = PurePosixPath(raw)
            if member.is_absolute() or ".." in member.parts or not member.parts:
                raise RuntimeError(f"Unsafe ZIP path rejected: {info.filename}")
            mode = (info.external_attr >> 16) & 0xFFFF
            if stat.S_ISLNK(mode):
                raise RuntimeError(f"Symlink rejected: {info.filename}")
            target = (destination / Path(*member.parts)).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"ZIP path escapes destination: {info.filename}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
            count += 1
            total_bytes += info.file_size
    return count, total_bytes

def make_release_bundles(files):
    BUNDLES.mkdir(parents=True, exist_ok=True)
    for old in BUNDLES.glob("*.zip"):
        old.unlink()
    ordered = sorted(files, key=lambda p: p.relative_to(EXTRACTED).as_posix())
    groups, current, current_size = [], [], 0
    for file_path in ordered:
        size = file_path.stat().st_size
        if size > MAX_BUNDLE_BYTES:
            raise RuntimeError(
                f"Cannot package {file_path.relative_to(EXTRACTED)} ({size:,} bytes): "
                "a single extracted file exceeds the safe per-asset bundle limit."
            )
        if current and current_size + size > MAX_BUNDLE_BYTES:
            groups.append(current)
            current, current_size = [], 0
        current.append(file_path)
        current_size += size
    if current:
        groups.append(current)

    bundles = []
    for index, group in enumerate(groups, 1):
        bundle = BUNDLES / f"extracted-assets-{index:03d}.zip"
        with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
            for file_path in group:
                archive.write(file_path, arcname=file_path.relative_to(EXTRACTED).as_posix())
        bundles.append({
            "path": bundle,
            "name": bundle.name,
            "size_bytes": bundle.stat().st_size,
            "sha256": sha256_file(bundle),
            "files": len(group),
        })
        print(f"Bundle ready: {bundle.name} | {bundle.stat().st_size:,} bytes | {len(group):,} files")
    return bundles

def github_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

def get_or_create_release(session):
    url = f"{GITHUB_API}/repos/{DEST_REPO}/releases/tags/{DEST_TAG}"
    response = session.get(url, timeout=60)
    if response.status_code == 200:
        release = response.json()
        print("Using existing release:", release["html_url"])
        return release
    if response.status_code != 404:
        response.raise_for_status()
    payload = {
        "tag_name": DEST_TAG,
        "name": DEST_RELEASE_NAME,
        "body": "Extracted contents of the verified Conquer Mobile source ZIP files. See extraction-report.json for checksums and file counts.",
        "draft": False,
        "prerelease": False,
    }
    created = session.post(
        f"{GITHUB_API}/repos/{DEST_REPO}/releases",
        json=payload,
        timeout=60,
    )
    created.raise_for_status()
    release = created.json()
    print("Created release:", release["html_url"])
    return release

def upload_release_asset(session, release, bundle):
    # Remove GitHub's URI template suffix from upload_url.
    upload_url = release["upload_url"].split("{", 1)[0]
    headers = github_headers(session.headers["Authorization"].split(" ", 1)[1])
    headers["Content-Type"] = "application/zip"
    with open(bundle["path"], "rb") as stream:
        response = requests.post(
            upload_url,
            params={"name": bundle["name"]},
            headers=headers,
            data=stream,
            timeout=(30, 1800),
        )
    # If rerunning, replace an asset with the same name instead of duplicating it.
    if response.status_code == 422 and "already_exists" in response.text:
        assets_url = f'{GITHUB_API}/repos/{DEST_REPO}/releases/{release["id"]}/assets'
        existing = session.get(assets_url, timeout=60)
        existing.raise_for_status()
        for asset in existing.json():
            if asset["name"] == bundle["name"]:
                deleted = session.delete(asset["url"], timeout=60)
                deleted.raise_for_status()
        with open(bundle["path"], "rb") as stream:
            response = requests.post(
                upload_url,
                params={"name": bundle["name"]},
                headers=headers,
                data=stream,
                timeout=(30, 1800),
            )
    response.raise_for_status()
    return response.json()

print("Checking available Colab disk space...")
BASE.mkdir(parents=True, exist_ok=True)
DOWNLOADS.mkdir(parents=True, exist_ok=True)
EXTRACTED.mkdir(parents=True, exist_ok=True)
required_download = sum(item["size"] for item in FILES)
free = shutil.disk_usage("/content").free
print(f"Free disk: {free:,} bytes | ZIP downloads required: {required_download:,} bytes")
if free < required_download + SAFETY_BYTES:
    raise RuntimeError("Not enough free disk space to download both ZIPs safely. Use a fresh Colab runtime.")

print("Downloading both ZIPs directly from the existing GitHub Release and verifying SHA-256...")
report = {
    "source_repo": SOURCE_REPO,
    "source_release_tag": SOURCE_TAG,
    "destination_repo": DEST_REPO,
    "destination_release_tag": DEST_TAG,
    "source_archives": {},
}
local_zips = {}
for item in FILES:
    local = DOWNLOADS / item["name"]
    size, digest = download_verified(item, local)
    local_zips[item["name"]] = local
    report["source_archives"][item["name"]] = {
        "size_bytes": size,
        "sha256": digest,
        "verified": True,
    }
    print(f"VERIFIED: {item['name']} ({size:,} bytes)")

print("Inspecting archive contents before extraction...")
total_uncompressed = 0
for name, local in local_zips.items():
    with zipfile.ZipFile(local) as archive:
        total_uncompressed += sum(info.file_size for info in archive.infolist() if not info.is_dir())
free = shutil.disk_usage("/content").free
print(f"Uncompressed data expected: {total_uncompressed:,} bytes | Current free disk: {free:,} bytes")
if free < total_uncompressed + SAFETY_BYTES:
    raise RuntimeError("Not enough free disk space for safe extraction. No extraction has started.")

print("Extracting with ZIP path traversal and symlink protections...")
for item in FILES:
    name = item["name"]
    target_dir = EXTRACTED / Path(name).stem
    target_dir.mkdir(parents=True, exist_ok=True)
    count, unpacked = safe_extract(local_zips[name], target_dir)
    report["source_archives"][name].update({
        "extracted_to": target_dir.relative_to(BASE).as_posix(),
        "extracted_files": count,
        "uncompressed_bytes": unpacked,
    })
    print(f"EXTRACTED: {name} -> {count:,} files ({unpacked:,} bytes)")

all_files = [p for p in EXTRACTED.rglob("*") if p.is_file()]
if not all_files:
    raise RuntimeError("Extraction produced no files; refusing to publish an empty release.")
report["total_extracted_files"] = len(all_files)
report["total_extracted_bytes"] = sum(p.stat().st_size for p in all_files)
report["status"] = "extraction_verified"

print("Packaging extracted files into GitHub Release assets...")
bundles = make_release_bundles(all_files)
report["release_assets"] = [
    {k: v for k, v in bundle.items() if k != "path"} for bundle in bundles
]

# Build a searchable inventory, a visual image contact sheet, and a lightweight HTML catalog.
import csv, html, mimetypes
from PIL import Image, ImageOps, ImageDraw
CATALOG_DIR = BASE / "catalog"
CATALOG_DIR.mkdir(parents=True, exist_ok=True)
INVENTORY_JSON = CATALOG_DIR / "asset-inventory.json"
INVENTORY_CSV = CATALOG_DIR / "asset-inventory.csv"
CONTACT_SHEET = CATALOG_DIR / "image-contact-sheet.jpg"
HTML_CATALOG = CATALOG_DIR / "asset-catalog.html"

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif", ".tif", ".tiff"}
MODEL_EXTS = {".glb", ".gltf", ".obj", ".fbx", ".dae", ".blend", ".3ds", ".stl", ".ply"}
AUDIO_EXTS = {".wav", ".mp3", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".webm", ".avi"}
TEXT_EXTS = {".json", ".xml", ".txt", ".ini", ".cfg", ".csv", ".md", ".tscn", ".tres", ".gd", ".shader"}

inventory = []
image_candidates = []
for p in sorted(EXTRACTED.rglob("*")):
    if not p.is_file():
        continue
    rel = p.relative_to(EXTRACTED).as_posix()
    ext = p.suffix.lower()
    kind = ("image" if ext in IMAGE_EXTS else
            "3d_model" if ext in MODEL_EXTS else
            "audio" if ext in AUDIO_EXTS else
            "video" if ext in VIDEO_EXTS else
            "text_or_config" if ext in TEXT_EXTS else "other")
    row = {"path": rel, "name": p.name, "extension": ext,
           "size_bytes": p.stat().st_size, "category": kind}
    if kind == "3d_model":
        row["preview_status"] = "indexed; requires a 3D viewer for visual inspection"
        if ext == ".obj":
            try:
                with open(p, "r", encoding="utf-8", errors="ignore") as fh:
                    vertices = faces = 0
                    for line in fh:
                        if line.startswith("v "): vertices += 1
                        elif line.startswith("f "): faces += 1
                row["obj_vertices"] = vertices
                row["obj_faces"] = faces
            except Exception as e:
                row["inspection_note"] = str(e)[:200]
        elif ext == ".gltf":
            try:
                data = json.loads(p.read_text(encoding="utf-8"))
                row["gltf_meshes"] = len(data.get("meshes", []))
                row["gltf_nodes"] = len(data.get("nodes", []))
                row["gltf_materials"] = len(data.get("materials", []))
            except Exception as e:
                row["inspection_note"] = str(e)[:200]
        elif ext == ".glb":
            try:
                with open(p, "rb") as fh:
                    header = fh.read(12)
                if len(header) == 12 and header[:4] == b"glTF":
                    row["glb_version"] = int.from_bytes(header[4:8], "little")
                    row["glb_declared_bytes"] = int.from_bytes(header[8:12], "little")
                else:
                    row["inspection_note"] = "Invalid GLB header"
            except Exception as e:
                row["inspection_note"] = str(e)[:200]
    inventory.append(row)
    if kind == "image":
        image_candidates.append(p)

INVENTORY_JSON.write_text(json.dumps(inventory, ensure_ascii=False, indent=2), encoding="utf-8")
with open(INVENTORY_CSV, "w", newline="", encoding="utf-8-sig") as fh:
    writer = csv.DictWriter(fh, fieldnames=sorted({k for row in inventory for k in row}))
    writer.writeheader()
    writer.writerows(inventory)

# Create contact sheet from up to 120 readable images; originals remain untouched.
thumbs = []
for p in image_candidates[:120]:
    try:
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((180, 130))
            tile = Image.new("RGB", (200, 170), "white")
            tile.paste(im, ((200-im.width)//2, 5))
            draw = ImageDraw.Draw(tile)
            label = p.name[:26]
            draw.text((6, 140), label, fill="black")
            thumbs.append(tile)
    except Exception:
        pass

if thumbs:
    cols = 4
    rows = (len(thumbs) + cols - 1) // cols
    sheet = Image.new("RGB", (cols*200, rows*170), "#dddddd")
    for i, tile in enumerate(thumbs):
        sheet.paste(tile, ((i % cols)*200, (i // cols)*170))
    sheet.save(CONTACT_SHEET, quality=88)
    contact_sheet_result = {"path": CONTACT_SHEET, "name": CONTACT_SHEET.name,
                            "size_bytes": CONTACT_SHEET.stat().st_size}
else:
    contact_sheet_result = None

# HTML is a portable index of all extracted assets and model inspection metadata.
html_rows = []
for row in inventory:
    html_rows.append("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(k, '')))}</td>"
        for k in ("category", "path", "size_bytes", "preview_status", "obj_vertices",
                  "obj_faces", "gltf_meshes", "gltf_nodes", "gltf_materials",
                  "glb_version", "inspection_note")
    ) + "</tr>")
headers = ("category", "path", "size_bytes", "preview_status", "obj_vertices",
           "obj_faces", "gltf_meshes", "gltf_nodes", "gltf_materials",
           "glb_version", "inspection_note")
HTML_CATALOG.write_text(
    "<!doctype html><meta charset='utf-8'><title>Conquer Mobile Asset Catalog</title>"
    "<h1>Conquer Mobile Asset Catalog</h1>"
    f"<p>Total files: {len(inventory)} | Images: {len(image_candidates)} | "
    f"3D models: {sum(1 for x in inventory if x['category']=='3d_model')}</p>"
    "<table border='1' cellspacing='0' cellpadding='5'><thead><tr>"
    + "".join(f"<th>{html.escape(h)}</th>" for h in headers)
    + "</tr></thead><tbody>" + "".join(html_rows) + "</tbody></table>",
    encoding="utf-8"
)

report["catalog"] = {
    "inventory_json": INVENTORY_JSON.name,
    "inventory_csv": INVENTORY_CSV.name,
    "html_catalog": HTML_CATALOG.name,
    "total_indexed_files": len(inventory),
    "image_files": len(image_candidates),
    "3d_model_files": sum(1 for x in inventory if x["category"] == "3d_model"),
    "contact_sheet": CONTACT_SHEET.name if contact_sheet_result else None,
    "note": "Images are represented in a contact sheet; 3D files are indexed and basic format metadata is inspected, not fully rendered."
}
print(f"Catalog created: {len(inventory):,} files | {len(image_candidates):,} images | "
      f"{report['catalog']['3d_model_files']:,} 3D models")
print("Catalog artifacts:", INVENTORY_JSON.name, INVENTORY_CSV.name, HTML_CATALOG.name,
      CONTACT_SHEET.name if contact_sheet_result else "(no readable images)")
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

token = userdata.get("GH_TOKEN")
if not token:
    raise RuntimeError(
        "Missing GH_TOKEN in Colab Secrets. Add a GitHub token with Contents: Read and write "
        "permission for ZzwahwahzZ/Conquer-Mobile-Assets., enable Notebook access, then rerun."
    )

session = requests.Session()
session.headers.update(github_headers(token))
who = session.get(f"{GITHUB_API}/user", timeout=60)
who.raise_for_status()
print("Authenticated to GitHub as:", who.json().get("login"))

# Verify the token can access the destination repository before uploading.
repo_check = session.get(f"{GITHUB_API}/repos/{DEST_REPO}", timeout=60)
repo_check.raise_for_status()

release = get_or_create_release(session)
uploaded_names = []
for bundle in bundles:
    print("Uploading:", bundle["name"])
    asset = upload_release_asset(session, release, bundle)
    if asset["name"] != bundle["name"] or asset["size"] != bundle["size_bytes"]:
        raise RuntimeError(f"GitHub asset verification failed for {bundle['name']}")
    uploaded_names.append(asset["name"])
    print(f"UPLOADED: {asset['name']} ({asset['size']:,} bytes)")

# Upload the machine-readable inventory, CSV, HTML catalog, optional image sheet, and report.
extra_assets = [
    {"path": INVENTORY_JSON, "name": INVENTORY_JSON.name, "size_bytes": INVENTORY_JSON.stat().st_size},
    {"path": INVENTORY_CSV, "name": INVENTORY_CSV.name, "size_bytes": INVENTORY_CSV.stat().st_size},
    {"path": HTML_CATALOG, "name": HTML_CATALOG.name, "size_bytes": HTML_CATALOG.stat().st_size},
]
if contact_sheet_result:
    extra_assets.append(contact_sheet_result)
extra_assets.append({"path": REPORT_PATH, "name": REPORT_PATH.name,
                     "size_bytes": REPORT_PATH.stat().st_size})
for extra in extra_assets:
    print("Uploading catalog/report asset:", extra["name"])
    uploaded = upload_release_asset(session, release, extra)
    if uploaded["name"] != extra["name"] or uploaded["size"] != extra["size_bytes"]:
        raise RuntimeError(f"Catalog/report upload verification failed: {extra['name']}")

# Re-read the release and verify every uploaded asset by exact name and size.
assets_response = session.get(
    f'{GITHUB_API}/repos/{DEST_REPO}/releases/{release["id"]}/assets',
    timeout=60,
)
assets_response.raise_for_status()
remote_assets = {asset["name"]: asset for asset in assets_response.json()}
for bundle in bundles:
    remote = remote_assets.get(bundle["name"])
    if not remote or remote["size"] != bundle["size_bytes"]:
        raise RuntimeError(f"Release verification failed for {bundle['name']}")
for extra in extra_assets:
    remote = remote_assets.get(extra["name"])
    if not remote or remote["size"] != extra["size_bytes"]:
        raise RuntimeError(f"Release catalog/report verification failed: {extra['name']}")

print("\nSUCCESS: GitHub source ZIPs downloaded, SHA-256 verified, safely extracted, repackaged,")
print("uploaded to a new GitHub Release, and verified against GitHub's asset listing.")
print("Release URL:", release["html_url"])
print("Extracted files:", f"{len(all_files):,}")
print("Extracted bytes:", f"{report['total_extracted_bytes']:,}")
print("Release bundles:", len(bundles))
